In [ ]:
# 1. zadatak

import numpy as np
import math
from numpy import linalg as la #ako vam treba
np.set_printoptions(precision=5, suppress=True)

def jedinicni(p):
  norm = la.norm(p)
  if not np.isclose(norm, 1.0):
        p = p / norm
  return p

def AxisAngle2A(pphi):
  phi = pphi[3]
  cosphi = math.cos(phi)
  sinphi = math.sin(phi)

  p = np.array([pphi[0], pphi[1], pphi[2]])
  p = jedinicni(p)
  p1, p2, p3 = p

  px = np.array([
      [0, -p3, p2],
      [p3, 0, -p1],
      [-p2, p1, 0]
  ])

  ppt = np.outer(p, p)

  E = np.eye(3)

  A = ppt + cosphi * (E - ppt) + sinphi * px

  A = np.where(np.isclose(A, 0) , 0.0 , A)  # izbegavanje -0. u rezultatu
  return A


pphi = np.array([1/3, -2/3,  2/3, np.pi/2])
print(AxisAngle2A(pphi))

[[ 0.11111 -0.88889 -0.44444]
 [ 0.44444  0.44444 -0.77778]
 [ 0.88889 -0.11111  0.44444]]


In [ ]:
# 2. zadatak

import numpy as np
import math
from numpy import linalg as la  #ako vam treba
np.set_printoptions(precision=5, suppress=True)

def Rx(phi):
  cos =  math.cos(phi)
  sin = math.sin(phi)
  return np.array([
      [1, 0, 0],
      [0, cos, -sin],
      [0, sin, cos]
  ])

def Ry(theta):
  cos =  math.cos(theta)
  sin = math.sin(theta)
  return np.array([
      [cos, 0, sin],
      [0, 1, 0],
      [-sin, 0, cos]
  ])

def Rz(ksi):
  cos =  math.cos(ksi)
  sin = math.sin(ksi)
  return np.array([
      [cos, -sin, 0],
      [sin, cos, 0],
      [0, 0, 1]
  ])

def Euler2A(uglovi):
  ksi, theta, phi = uglovi

  A = Rz(ksi) @ Ry(theta) @ Rx(phi)

  A = np.where(np.isclose(A, 0) , 0 , A)  # da bi izbegli -0. u rezultatu
  return A


In [ ]:
uglovi = np.array([np.pi/2, -np.pi/4, (7/8)*np.pi])
print(Euler2A(uglovi))

[[ 0.       0.92388  0.38268]
 [ 0.70711 -0.2706   0.65328]
 [ 0.70711  0.2706  -0.65328]]


In [ ]:
# 3. zadatak (nisu mi prosli svi primeri imam 1.35/1.5)

import numpy as np
import math
from numpy import linalg as la  #ako vam treba
np.set_printoptions(precision=5, suppress=True)

def jeste_matrica_kretanja(A):
  if A.shape != (3, 3):
    return False

  if not np.allclose(A @ A.T, np.eye(3), atol=1e-6) or not np.isclose(la.det(A), 1.0, atol=1e-6):
    return False

  return True

def vektor_ugao(A):
  M = A - np.eye(3)

  u = M[:, 0]
  v = M[:, 1]

  if np.isclose(la.norm(np.cross(u, v)), 0):
        v = M[:, 2]

  p = np.cross(u, v)
  p = p / la.norm(p)

  u = u / la.norm(u)
  u1 = A @ u
  cos_phi = np.dot(u, u1) / la.norm(u1)
  phi = math.acos(cos_phi)

  if np.dot(u, np.cross(u1, p)) < 0:
    p = -p

  return p, phi

def A2AxisAngle(A):
  if not jeste_matrica_kretanja(A):
    return "Nije matrica kretanja!"

  if np.allclose(A, np.eye(3), atol=1e-6):
      return (np.array([1, 0, 0]), 0)

  p, phi = vektor_ugao(A)
  px, py, pz = p

  pphi = np.array([px,py,pz,phi])  # osa i ugao idu u jedan vektor
  pphi = np.where(np.isclose(pphi, 0) , 0 , pphi)  # izbegavanje -0. u rezultatu
  return pphi

In [ ]:
A = (1/9)*np.array([[1,-8,-4], [4,4,-7], [8,-1,4]])
print(A2AxisAngle(A))

[ 0.33333 -0.66667  0.66667  1.5708 ]


In [ ]:
A = (1/9)*np.array([[1,-8,-4], [4,4,-7], [8,-1,5]])  #primetite 4->5, matrica A nije ortogonalna
print(A2AxisAngle(A))

Nije matrica kretanja!


In [ ]:
A = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])
print(A2AxisAngle(A))

[0.57735 0.57735 0.57735 2.0944 ]


In [ ]:
A = np.array([[1,0,0], [0,1,0], [0,0,1]])
print(A2AxisAngle(A))

(array([1, 0, 0]), 0)


In [ ]:
A = np.array([[0,1,0], [0,0,1], [1,0,0]]) # uporediti sa primerom za Rodrigeza sa casa, nije bas isto
print(A2AxisAngle(A))

[-0.57735 -0.57735 -0.57735  2.0944 ]


In [ ]:
# 4. zadatak

import numpy as np
import math
from numpy import linalg as la #ako vam treba
np.set_printoptions(precision=5, suppress=True)  # formatiranje izlaza na 5 decimala

def jeste_matrica_kretanja(A):
  if A.shape != (3, 3):
    return False

  if not np.allclose(A @ A.T, np.eye(3), atol=1e-6) or not np.isclose(la.det(A), 1.0, atol=1e-6):
    return False

  return True

def A2Euler(A):
  if not jeste_matrica_kretanja(A):
    return "Nije matrica kretanja!"

  a11, a12, a13 = A[0, 0], A[0, 1], A[0, 2]
  a21, a22, a23 = A[1, 0], A[1, 1], A[1, 2]
  a31, a32, a33 = A[2, 0], A[2, 1], A[2, 2]

  if a31 < 1:
    if a31 > -1:
      psi = math.atan2(a21, a11)
      theta = math.asin(-a31)
      phi = math.atan2(a32, a33)
    else:
      psi = math.atan2(-a12, a22)
      theta = math.pi / 2
      phi = 0
  else:
    psi = math.atan2(-a12, a22)
    theta = -math.pi / 2
    phi = 0

  psi = (psi + math.pi) % (2 * math.pi) - math.pi
  phi = (phi + math.pi) % (2 * math.pi) - math.pi

  uglovi = np.array([psi, theta, phi])
  uglovi = np.where(np.isclose(uglovi, 0) , 0 , uglovi)
  return uglovi

In [ ]:
A = (1/9)*np.array([[1,-8,-4], [4,4,-7], [8,-1,4]])
print(A2Euler(A))

[ 1.32582 -1.09491 -0.24498]


In [ ]:
A = (1/9)*np.array([[1,-8,-4], [4,4,-7], [8,-1,5]])  #primetite 4->5, matrica A nije ortogonalna
print(A2Euler(A))

Nije matrica kretanja!


In [ ]:
# zakljucan ziroskop
A = np.array([[0,1,0], [0,0,1], [1,0,0]])
print(A2Euler(A))

[-1.5708 -1.5708  0.    ]
